# Lab 08: Challenge - Production-Ready AI API

**Goal:** Build a complete production-ready FastAPI application combining ALL patterns from this session: FastAPI endpoints, Pydantic models, health checks, structured logging, secrets management, and production checklist validation.

**Scenario:**

UniGPS is deploying an AI support agent to production. You need to create:
1. A FastAPI app with Pydantic request/response models
2. A production-grade /health endpoint with dependency checks
3. Structured JSON logging configuration
4. A .env file and Python deployment config (load_dotenv + uvicorn)
5. A production readiness checklist score

No API key needed - pure Python + FastAPI.

In [ ]:
import os
import shutil
import json
import textwrap
from datetime import datetime

WORKDIR = "/tmp/prod-lab-11-08"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

## Architecture Overview

```
Production AI API Stack (Python):
==================================
  Client -> uvicorn -> FastAPI (+ /health)
                         |
                   +-----+-----+
                   |           |
                LangGraph   ChromaDB
                 Agent    (in-process)
                   |
                Groq API
                (.env + load_dotenv)
```

**Production layers:**
1. **API:** FastAPI + Pydantic validation
2. **Health:** /health with dependency checks + HealthChecker
3. **Logging:** Structured JSON with trace_id
4. **Secrets:** .env file + load_dotenv() + uvicorn launch
5. **Checklist:** Readiness score

## TODO 1: FastAPI App with Pydantic Models

Create a FastAPI app with:
- **SupportRequest** model: `employee_name` (str, min 2 chars), `request` (str, min 5 chars), `priority` (str, default='normal', pattern: low|normal|high|urgent)
- **SupportResponse** model: `category` (str), `response` (str), `priority` (str), `timestamp` (str)
- POST `/api/support` endpoint using the models
- Simple keyword classifier for: hr, tech, finance, general

In [ ]:
todo1_code = textwrap.dedent("""\
    # TODO: Production app with API framework and data validation
    # 1. Import the web framework, model base class, and field validator
    # 2. Create request model with field constraints
    # 3. Create response model
    # 4. Add category detection function
    # 5. Add POST endpoint at the support path

""")

with open(os.path.join(WORKDIR, "app.py"), "w") as f:
    f.write(todo1_code)

In [ ]:
checks1 = [
    ("Has FastAPI import",       "FastAPI" in todo1_code or "fastapi" in todo1_code),
    ("Has BaseModel",            "BaseModel" in todo1_code),
    ("Has Field import",         "Field" in todo1_code),
    ("Has SupportRequest",       "SupportRequest" in todo1_code),
    ("Has SupportResponse",      "SupportResponse" in todo1_code),
    ("Has employee_name",        "employee_name" in todo1_code),
    ("Has min_length",           "min_length" in todo1_code),
    ("Has priority pattern",     "pattern" in todo1_code or "priority" in todo1_code),
    ("Has /api/support",         "/api/support" in todo1_code),
    ("Has classify function",    "classify" in todo1_code),
]

score1 = sum(1 for _, ok in checks1 if ok)
print(f"Validating ({score1}/{len(checks1)}):\n")
for name, ok in checks1:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

## TODO 2: Integrated Health Endpoint with Structured Logging

Unlike Lab 03 which focused on the /health endpoint pattern alone, this challenge integrates health checking with structured JSON logging (from Lab 06). Create a `/health` endpoint that:
- Checks redis (`redis_client.ping()`)
- Checks chromadb (`chroma_client.heartbeat()`)
- Checks model (`agent.is_ready()`)
- Returns 200 with `{"status": "ok"}` if all healthy
- Returns 503 with `{"status": "degraded"}` if any check fails
- Include a `checks` dict showing each dependency status
- **Integration:** Log each health check result using structured JSON logging with `trace_id` correlation

In [ ]:
todo2_code = textwrap.dedent("""\
    # TODO: Production readiness endpoint
    # 1. Add GET route for liveness/readiness probe
    # 2. Check all dependencies (cache, vector DB, LLM)
    # 3. Return appropriate HTTP response code based on results

""")

with open(os.path.join(WORKDIR, "health.py"), "w") as f:
    f.write(todo2_code)

In [ ]:
checks2 = [
    ("Has /health route",       "/health" in todo2_code),
    ("Has async def",           "async def" in todo2_code or "def health" in todo2_code),
    ("Checks redis",            "redis" in todo2_code),
    ("Checks chromadb",         "chromadb" in todo2_code or "chroma" in todo2_code),
    ("Checks model",            "model" in todo2_code or "agent" in todo2_code or "ready" in todo2_code),
    ("Returns 200 or 503",      "200" in todo2_code and "503" in todo2_code),
    ("Has status field",        "status" in todo2_code),
    ("Has checks dict",         "checks" in todo2_code),
]

score2 = sum(1 for _, ok in checks2 if ok)
print(f"Validating ({score2}/{len(checks2)}):\n")
for name, ok in checks2:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

## TODO 3: Structured JSON Logging with AI-Specific Fields

Unlike Lab 06 which covered the JSONFormatter pattern, this challenge requires creating a complete logging config that integrates with your FastAPI app and health endpoint. Create a Python logging config that outputs JSON with:
- `timestamp`, `level`, `message`, `logger` fields
- Extra fields: `trace_id`, `user_id`, `model`, `tokens_in`, `tokens_out`, `cost_usd`, `duration_s`
- A `JSONFormatter` class and logger setup
- Example log entry as a JSON string showing an actual AI request with all fields populated

In [ ]:
todo3_code = textwrap.dedent("""\
    # TODO: Structured output for log aggregation
    # 1. Create a custom class to format records as JSON
    # 2. Wire up stream output and named log instance
    # 3. Write an example entry with extra fields

""")

with open(os.path.join(WORKDIR, "logging_config.py"), "w") as f:
    f.write(todo3_code)

# Also check for a sample JSON log entry
todo3_log = "___"

In [ ]:
checks3 = [
    ("Has JSONFormatter class",  "JSONFormatter" in todo3_code or "Formatter" in todo3_code),
    ("Has logging import",       "logging" in todo3_code or "import json" in todo3_code),
    ("Has trace_id field",       "trace_id" in todo3_code),
    ("Has handler setup",        "handler" in todo3_code or "Handler" in todo3_code),
    ("Has logger setup",         "logger" in todo3_code or "getLogger" in todo3_code),
]

# Validate the sample log entry
log_valid = False
if todo3_log != "___":
    try:
        parsed = json.loads(todo3_log)
        log_valid = all(
            any(f in k.lower() for k in parsed.keys())
            for f in ["level", "message"]
        )
    except (json.JSONDecodeError, TypeError):
        log_valid = False

checks3.append(("Sample log is valid JSON", log_valid))

score3 = sum(1 for _, ok in checks3 if ok)
print(f"Validating ({score3}/{len(checks3)}):\n")
for name, ok in checks3:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

## TODO 4: Integrated Secrets Management with Deployment Config

Unlike Lab 05 which introduced .env files and load_dotenv() individually, this challenge combines secrets management with a complete deployment config. Create:
- A `.env` file with: `GROQ_API_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_PUBLIC_KEY`
- Python code using `load_dotenv()` to load `.env` and validate required keys at startup
- A `.gitignore` entry for `.env`
- A uvicorn launch command that ties together the app, health endpoint, and logging config
- A signal handler for graceful shutdown (flush logs, close connections)

In [ ]:
todo4_config = textwrap.dedent("""\
    # TODO: Environment file and deployment configuration
    # 1. Define environment variables for API keys
    # 2. Load environment from file using Python
    # 3. Add graceful shutdown handling
    # 4. Configure ASGI server launch command

""")

with open(os.path.join(WORKDIR, "deploy_config.py"), "w") as f:
    f.write(todo4_config)

In [ ]:
checks4 = [
    ("Has .env content",        ".env" in todo4_config or "GROQ" in todo4_config),
    ("Has GROQ_API_KEY",        "GROQ_API_KEY" in todo4_config),
    ("Has LANGFUSE_SECRET_KEY", "LANGFUSE_SECRET_KEY" in todo4_config),
    ("Has LANGFUSE_PUBLIC_KEY", "LANGFUSE_PUBLIC_KEY" in todo4_config),
    ("Has load_dotenv",         "load_dotenv" in todo4_config),
    ("Has uvicorn",             "uvicorn" in todo4_config),
    ("Has signal handler",      "signal" in todo4_config or "SIGTERM" in todo4_config),
    ("Has health check",        "health" in todo4_config.lower()),
]

score4 = sum(1 for _, ok in checks4 if ok)
print(f"Validating ({score4}/{len(checks4)}):\n")
for name, ok in checks4:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

## TODO 5: Production Checklist

Score your production readiness (answer each with `"yes"` or `"no"`):

In [ ]:
checklist = [
    {
        "item": "Health endpoint checks all dependencies (redis, chromadb, model)",
        "answer": "___",
        "correct": "yes",
    },
    {
        "item": "Structured JSON logging with trace_id correlation",
        "answer": "___",
        "correct": "yes",
    },
    {
        "item": "API keys stored in .env file (not hardcoded or in git)",
        "answer": "___",
        "correct": "yes",
    },
    {
        "item": "Pydantic models validate all request inputs",
        "answer": "___",
        "correct": "yes",
    },
    {
        "item": "Signal handler (SIGTERM) configured for graceful shutdown",
        "answer": "___",
        "correct": "yes",
    },
    {
        "item": "psutil monitoring for memory/CPU to prevent OOM",
        "answer": "___",
        "correct": "yes",
    },
]

# YOUR CODE HERE: Answer "yes" or "no" for each item
# checklist[0]["answer"] = "yes"

In [ ]:
score5 = 0
for i, c in enumerate(checklist, 1):
    if c["answer"] == "___":
        status = "TODO"
    elif c["answer"].strip().lower() == c["correct"]:
        status = "PASS"
        score5 += 1
    else:
        status = "FAIL"
    print(f"  [{status}] {i}. {c['item']}")

print(f"\nChecklist score: {score5}/{len(checklist)}")

## Challenge Summary

In [ ]:
total_checks = len(checks1) + len(checks2) + len(checks3) + len(checks4) + len(checklist)
total_score = score1 + score2 + score3 + score4 + score5

print(f"TODO 1 - FastAPI + Pydantic:       {score1}/{len(checks1)}")
print(f"TODO 2 - Health Endpoint:           {score2}/{len(checks2)}")
print(f"TODO 3 - Structured Logging:        {score3}/{len(checks3)}")
print(f"TODO 4 - .env + Python Deploy:      {score4}/{len(checks4)}")
print(f"TODO 5 - Production Checklist:      {score5}/{len(checklist)}")
print(f"\nTOTAL: {total_score}/{total_checks}")
print(f"\nFiles generated in {WORKDIR}/")
print(f"  - app.py              (FastAPI application)")
print(f"  - health.py           (Health endpoint)")
print(f"  - logging_config.py   (JSON logging setup)")
print(f"  - deploy_config.py    (Python deployment config)")

if total_score == total_checks:
    print(f"\nPRODUCTION READY! All checks passed.")
elif total_score >= total_checks * 0.7:
    print(f"\nALMOST THERE! Fix remaining items for production readiness.")
else:
    print(f"\nNEEDS WORK. Review the session labs and fill in TODO sections.")

print(f"\nCheck solutions/lab08_challenge.ipynb when done!")